# 02 · Ricostruzione temporale

La lavagna è ferma, il docente no. Per ogni pixel della lavagna rettificata si tiene la **mediana degli
ultimi N campioni in cui non era coperto**. Ne escono:

- la lavagna pulita in ogni istante (snapshot quando il contenuto cambia, cioè scrittura o cancellazione);
- il punto indicato dal docente in coordinate della lavagna, con timestamp, da allineare alla trascrizione.

Senza un video reale si usa una sequenza sintetica: sfondo ricavato dal frame (inpainting della docente)
e sagoma che scorre davanti alla lavagna. Con un video vero basta la CLI:

```
lavagna data/video/lezione.mp4 --step 1 --crop 0,68,1674,783
```

In [ ]:
from pathlib import Path
import cv2, numpy as np, matplotlib.pyplot as plt
from image_segmentation.board import detect_board, rectify_homography
from image_segmentation.person import PersonSegmenter
from image_segmentation.reconstruction import BoardReconstructor

ROOT = Path.cwd().parent
x, y, w, h = 0, 68, 1674, 783
frame = cv2.imread(str(ROOT / "data/frame/lezione_prova_cit_censured.png"))[y:y+h, x:x+w]
segment = PersonSegmenter()
person = segment(frame)
background = cv2.inpaint(frame, person.mask, 7, cv2.INPAINT_TELEA)

def paste(dx):
    M = np.float32([[1, 0, dx], [0, 1, 0]])
    moved = cv2.warpAffine(frame, M, (w, h))
    m = cv2.warpAffine(cv2.erode(person.mask, np.ones((15, 15), np.uint8)), M, (w, h)) > 0
    out = background.copy(); out[m] = moved[m]; return out

offsets = [0, 250, 500, 750, 1000, 750, 500, 250, 0, -250]
frames = [paste(dx) for dx in offsets]
quad = detect_board(frame, person.mask)
H, size = rectify_homography(quad)

In [ ]:
rec = BoardReconstructor(H, size, window=5)
track = []
for f in frames:
    p = segment(f)
    rec.update(f, p.mask)
    track.append(rec.to_board(p.pointer))

truth = cv2.warpPerspective(background, H, size).astype(np.float32)
err = lambda img: float((np.abs(img.astype(np.float32) - truth).max(axis=2) > 40).mean())
print(f"pixel errati: frame singolo {err(cv2.warpPerspective(frames[-1], H, size)):.4f}, ricostruzione {err(rec.board):.4f}")

fig, axs = plt.subplots(2, 1, figsize=(14, 8))
axs[0].imshow(cv2.cvtColor(cv2.warpPerspective(frames[-1], H, size), cv2.COLOR_BGR2RGB)); axs[0].set_title("frame singolo")
axs[1].imshow(cv2.cvtColor(rec.image(), cv2.COLOR_BGR2RGB)); axs[1].set_title("ricostruzione + punti indicati")
pts = np.array([t for t in track if t is not None])
axs[1].plot(pts[:, 0], pts[:, 1], "o-", color="lime")
for a in axs: a.axis("off")
plt.tight_layout(); plt.show()

## Da verificare su un video reale

- **Ombre** del docente sulla lavagna: non sono nella maschera della persona.
- **Scrittura**: avviene sotto la mano, quindi compare nella ricostruzione solo quando la mano si sposta (ritardo circa finestra × passo).
- **Soglia degli snapshot** (`--min-change`): tarare su una lezione intera.
- **Punto indicato**: distinguere l'indicare dal gesticolare (`arm_angle`, durata della posa).